In [1]:
import os
for f in ["ai_generated.csv", "human_text.csv", "full_dataset.csv"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")

Deleted human_text.csv


In [2]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="Hello-SimpleAI/HC3", repo_type="dataset", local_dir="./hc3_data")

/sfs/weka/scratch/hjd3db/project-1/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 1266.06it/s]


'/sfs/weka/scratch/hjd3db/project-1/hc3_data'

In [3]:
import json
import pandas as pd

with open("./hc3_data/all.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

questions = [row['question'] for row in data[:250]]
human_texts = [row['human_answers'][0] for row in data[:250] if row['human_answers']]

human_df = pd.DataFrame({
    "question": questions[:len(human_texts)],
    "text": human_texts,
    "label": 0
})
human_df.to_csv("human_text.csv", index=False)
print(f"Human samples: {len(human_df)}")

Human samples: 250


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:09<00:00, 31.17it/s]


In [5]:
from tqdm import tqdm

SYSTEM_PROMPT = """You are a person casually answering a question online. 
Write naturally like a human — vary your sentence length, use informal language, 
occasionally be imperfect, and avoid sounding structured or overly formal. 
Write 150-250 words."""

results = []

for i, question in enumerate(tqdm(questions)):
    prompt = f"[INST] {SYSTEM_PROMPT}\n\n{question} [/INST]"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.9,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )

    results.append({"question": question, "text": response, "label": 1})

    if i % 100 == 0:
        pd.DataFrame(results).to_csv("ai_generated.csv", index=False)
        print(f"Saved {i} samples")

  0%|          | 1/250 [00:08<36:13,  8.73s/it]

Saved 0 samples


 40%|████      | 101/250 [12:59<17:55,  7.22s/it]

Saved 100 samples


 80%|████████  | 201/250 [25:46<06:25,  7.87s/it]

Saved 200 samples


100%|██████████| 250/250 [31:56<00:00,  7.67s/it]


In [6]:
ai_df = pd.DataFrame(results)
ai_df.to_csv("ai_generated.csv", index=False)

full_dataset = pd.concat([ai_df, human_df]).sample(frac=1).reset_index(drop=True)
full_dataset.to_csv("full_dataset.csv", index=False)
print(f"Done — {len(full_dataset)} total samples")

Done — 500 total samples
